# StockSage Multi-Stock Deep Learning Training

This notebook trains the offline CNN-LSTM direction model used by `app.py`. The app still trains ticker-local sklearn ensembles at runtime, but this notebook produces the reusable deep-learning sequence model that can be blended into the short-horizon signal.


In [8]:
import json
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import yfinance as yf
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cpu')

In [9]:
# Train on a broad, liquid universe so the LSTM learns cross-stock patterns.
# You can add/remove tickers, but keep the list liquid and avoid very short histories.
TICKERS = [
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', 'AMD', 'NFLX', 'AVGO',
    'JPM', 'BAC', 'GS', 'V', 'MA', 'COST', 'WMT', 'HD', 'UNH', 'LLY',
    'XOM', 'CVX', 'CAT', 'BA', 'GE', 'PLTR', 'SHOP', 'CRM', 'ADBE', 'ORCL',
    'SPY', 'QQQ', 'IWM', 'DIA'
]

MARKET_TICKERS = ['SPY', 'QQQ', '^VIX']
START_DATE = '2015-01-01'
SEQ_LEN = 30
DIRECTION_TARGET_DAYS = 5
BATCH_SIZE = 128
EPOCHS = 35
PATIENCE = 6
MODEL_DIR = Path('model_artifacts')
MODEL_DIR.mkdir(exist_ok=True)

FEATURE_COLS = [
    'Returns', 'Log_Returns', 'MA20_Dist', 'MA50_Dist', 'RSI', 'Vol_Rel',
    'Volatility20', 'Momentum5', 'Momentum10', 'Momentum20', 'MACD', 'MACD_Hist',
    'Range_Pct', 'SPY_Returns', 'QQQ_Returns', 'VIX_Returns', 'Rel_SPY_5',
    'Rel_QQQ_5', 'SPY_Momentum20', 'QQQ_Momentum20', 'VIX_Level',
]

len(FEATURE_COLS)


21

In [10]:
def normalize_price_index(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    return df[~df.index.duplicated(keep='last')]


def download_ohlcv(ticker, start=START_DATE):
    df = yf.download(ticker, start=start, auto_adjust=False, progress=False)
    if df.empty:
        raise ValueError(f'No data returned for {ticker}')
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    required = ['Open', 'High', 'Low', 'Close', 'Volume']
    return normalize_price_index(df[required].dropna())


def download_market_context(start=START_DATE):
    market = yf.download(MARKET_TICKERS, start=start, auto_adjust=False, progress=False)
    close = market['Close'] if isinstance(market.columns, pd.MultiIndex) else market[['Close']]
    close = normalize_price_index(close)

    context = pd.DataFrame(index=close.index)
    context['SPY_Returns'] = close['SPY'].pct_change()
    context['QQQ_Returns'] = close['QQQ'].pct_change()
    context['VIX_Returns'] = close['^VIX'].pct_change()
    context['SPY_Momentum20'] = close['SPY'] / close['SPY'].shift(20) - 1
    context['QQQ_Momentum20'] = close['QQQ'] / close['QQQ'].shift(20) - 1
    context['VIX_Level'] = close['^VIX']
    return context.replace([np.inf, -np.inf], np.nan).ffill()


def add_features(df, market_context):
    df = normalize_price_index(df).join(market_context, how='left')
    market_cols = ['SPY_Returns', 'QQQ_Returns', 'VIX_Returns', 'SPY_Momentum20', 'QQQ_Momentum20', 'VIX_Level']
    df[market_cols] = df[market_cols].ffill()

    df['Returns'] = df['Close'].pct_change()
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()
    df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()

    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))

    df['Vol_Rel'] = df['Volume'] / df['Volume'].rolling(20).mean()
    df['Volatility20'] = df['Returns'].rolling(20).std()
    df['Momentum5'] = df['Close'] / df['Close'].shift(5) - 1
    df['Momentum10'] = df['Close'] / df['Close'].shift(10) - 1
    df['Momentum20'] = df['Close'] / df['Close'].shift(20) - 1
    df['MA20_Dist'] = df['Close'] / df['MA20'] - 1
    df['MA50_Dist'] = df['Close'] / df['MA50'] - 1
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']
    df['Range_Pct'] = (df['High'] - df['Low']) / df['Close']
    df['Rel_SPY_5'] = df['Close'].pct_change(5) - df['SPY_Returns'].rolling(5).sum()
    df['Rel_QQQ_5'] = df['Close'].pct_change(5) - df['QQQ_Returns'].rolling(5).sum()
    df['Target'] = (df['Close'].shift(-DIRECTION_TARGET_DAYS) > df['Close']).astype(int)
    df['Forward_Log_Return'] = np.log(df['Close'].shift(-DIRECTION_TARGET_DAYS) / df['Close'])

    return df.replace([np.inf, -np.inf], np.nan).dropna()


In [11]:
market_context = download_market_context()
stock_frames = {}
failed = []

for ticker in TICKERS:
    try:
        raw = download_ohlcv(ticker)
        feat = add_features(raw, market_context)
        if len(feat) < 500:
            raise ValueError(f'Only {len(feat)} usable rows')
        stock_frames[ticker] = feat
        print(f'{ticker:>5}: {len(feat):4d} rows')
    except Exception as exc:
        failed.append((ticker, str(exc)))
        print(f'{ticker:>5}: skipped ({exc})')

print(f'Loaded {len(stock_frames)} tickers, failed {len(failed)}')
failed[:5]


 AAPL: 2802 rows
 MSFT: 2802 rows
 NVDA: 2802 rows
 AMZN: 2802 rows
GOOGL: 2802 rows
 META: 2802 rows
 TSLA: 2802 rows
  AMD: 2802 rows
 NFLX: 2802 rows
 AVGO: 2802 rows
  JPM: 2802 rows
  BAC: 2802 rows
   GS: 2802 rows
    V: 2802 rows
   MA: 2802 rows
 COST: 2802 rows
  WMT: 2802 rows
   HD: 2802 rows
  UNH: 2802 rows
  LLY: 2802 rows
  XOM: 2802 rows
  CVX: 2802 rows
  CAT: 2802 rows
   BA: 2802 rows
   GE: 2802 rows
 PLTR: 1356 rows
 SHOP: 2707 rows
  CRM: 2802 rows
 ADBE: 2802 rows
 ORCL: 2802 rows
  SPY: 2802 rows
  QQQ: 2802 rows
  IWM: 2802 rows
  DIA: 2802 rows
Loaded 34 tickers, failed 0


[]

In [12]:
# Chronological per-ticker split. This prevents future samples from leaking into training.
def split_ticker_frame(df, train_frac=0.70, valid_frac=0.15):
    n = len(df)
    train_end = int(n * train_frac)
    valid_end = int(n * (train_frac + valid_frac))
    return train_end, valid_end

train_feature_rows = []
splits = {}
for ticker, df in stock_frames.items():
    train_end, valid_end = split_ticker_frame(df)
    splits[ticker] = (train_end, valid_end)
    train_feature_rows.append(df.iloc[:train_end][FEATURE_COLS])

scaler = RobustScaler()
scaler.fit(pd.concat(train_feature_rows, axis=0).astype(float))

print('Scaler fitted on train-only rows:', sum(len(x) for x in train_feature_rows))


Scaler fitted on train-only rows: 65595


In [13]:
def make_sequences_for_ticker(ticker, df):
    train_end, valid_end = splits[ticker]
    values = scaler.transform(df[FEATURE_COLS].astype(float))
    target = df['Target'].to_numpy(dtype=np.float32)
    dates = df.index.to_numpy()

    buckets = {'train': [], 'valid': [], 'test': []}
    for start in range(0, len(df) - SEQ_LEN - DIRECTION_TARGET_DAYS + 1):
        end = start + SEQ_LEN
        target_idx = end - 1
        x = values[start:end]
        y = target[target_idx]

        if target_idx < train_end:
            split = 'train'
        elif target_idx < valid_end:
            split = 'valid'
        else:
            split = 'test'

        buckets[split].append((x, y, ticker, dates[target_idx]))
    return buckets

all_buckets = {'train': [], 'valid': [], 'test': []}
for ticker, df in stock_frames.items():
    buckets = make_sequences_for_ticker(ticker, df)
    for split_name in all_buckets:
        all_buckets[split_name].extend(buckets[split_name])

for split_name, rows in all_buckets.items():
    labels = np.array([r[1] for r in rows])
    print(f'{split_name:>5}: {len(rows):6d} samples | bullish rate {labels.mean():.2%}')


train:  64609 samples | bullish rate 55.91%
valid:  14049 samples | bullish rate 57.84%
 test:  13913 samples | bullish rate 54.82%


In [14]:
def bucket_to_tensors(rows):
    X = np.stack([r[0] for r in rows]).astype(np.float32)
    y = np.array([r[1] for r in rows], dtype=np.float32).reshape(-1, 1)
    return torch.tensor(X), torch.tensor(y)

X_train, y_train = bucket_to_tensors(all_buckets['train'])
X_valid, y_valid = bucket_to_tensors(all_buckets['valid'])
X_test, y_test = bucket_to_tensors(all_buckets['test'])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

X_train.shape, X_valid.shape, X_test.shape


(torch.Size([64609, 30, 21]),
 torch.Size([14049, 30, 21]),
 torch.Size([13913, 30, 21]))

In [15]:
class StockSageModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.cnn = nn.Conv1d(in_channels=n_features, out_channels=64, kernel_size=3)
        self.lstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.cnn(x))
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        x = self.dropout(self.relu(self.fc1(out[:, -1, :])))
        return self.fc2(x)

model = StockSageModel(len(FEATURE_COLS)).to(device)
model


StockSageModel(
  (cnn): Conv1d(21, 64, kernel_size=(3,), stride=(1,))
  (lstm): LSTM(64, 64, batch_first=True)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
)

In [16]:
def evaluate(model, loader):
    model.eval()
    losses, probs, labels = [], [], []
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            losses.append(loss.item() * len(xb))
            probs.extend(torch.sigmoid(logits).cpu().numpy().ravel())
            labels.extend(yb.cpu().numpy().ravel())

    probs = np.array(probs)
    labels = np.array(labels)
    preds = (probs >= 0.5).astype(int)
    metrics = {
        'loss': float(np.sum(losses) / len(labels)),
        'accuracy': float(accuracy_score(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'auc': float(roc_auc_score(labels, probs)) if len(np.unique(labels)) > 1 else 0.5,
        'baseline_accuracy': float(max(labels.mean(), 1 - labels.mean())),
    }
    return metrics

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

best_valid_loss = float('inf')
best_state = None
patience_left = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(xb)

    train_loss /= len(X_train)
    valid_metrics = evaluate(model, valid_loader)
    scheduler.step(valid_metrics['loss'])

    row = {'epoch': epoch, 'train_loss': train_loss, **{f'valid_{k}': v for k, v in valid_metrics.items()}}
    history.append(row)
    print(
        f"Epoch {epoch:02d} | train {train_loss:.4f} | valid loss {valid_metrics['loss']:.4f} "
        f"| acc {valid_metrics['accuracy']:.3f} | auc {valid_metrics['auc']:.3f} "
        f"| base {valid_metrics['baseline_accuracy']:.3f}"
    )

    if valid_metrics['loss'] < best_valid_loss:
        best_valid_loss = valid_metrics['loss']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_left = PATIENCE
    else:
        patience_left -= 1
        if patience_left == 0:
            print('Early stopping triggered')
            break

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.tail()


Epoch 01 | train 0.6828 | valid loss 0.6904 | acc 0.540 | auc 0.514 | base 0.578
Epoch 02 | train 0.6498 | valid loss 0.7055 | acc 0.540 | auc 0.544 | base 0.578
Epoch 03 | train 0.6011 | valid loss 0.7467 | acc 0.524 | auc 0.520 | base 0.578
Epoch 04 | train 0.5609 | valid loss 0.8213 | acc 0.518 | auc 0.512 | base 0.578
Epoch 05 | train 0.5191 | valid loss 0.8495 | acc 0.510 | auc 0.517 | base 0.578
Epoch 06 | train 0.5013 | valid loss 0.8960 | acc 0.514 | auc 0.504 | base 0.578
Epoch 07 | train 0.4866 | valid loss 0.9107 | acc 0.516 | auc 0.515 | base 0.578
Early stopping triggered


,epoch,train_loss,valid_loss,valid_accuracy,valid_precision,valid_recall,valid_auc,valid_baseline_accuracy
2,3,0.601059,0.746733,0.523738,0.590127,0.578144,0.519612,0.578404
3,4,0.560901,0.821285,0.517688,0.586739,0.561900,0.512373,0.578404
4,5,0.519099,0.849506,0.509787,0.585720,0.520921,0.517333,0.578404
5,6,0.501304,0.896042,0.513916,0.583948,0.555132,0.504397,0.578404
6,7,0.486581,0.910683,0.515624,0.588385,0.541103,0.515459,0.578404


In [18]:
valid_metrics = evaluate(model, valid_loader)
test_metrics = evaluate(model, test_loader)

print('Validation metrics')
print(json.dumps(valid_metrics, indent=2))
print('Test metrics')
print(json.dumps(test_metrics, indent=2))


Validation metrics
{
  "loss": 0.6904228844543532,
  "accuracy": 0.540180795786177,
  "precision": 0.5899373785359534,
  "recall": 0.6724095495938961,
  "auc": 0.5139799051316907,
  "baseline_accuracy": 0.5784041285514832
}
Test metrics
{
  "loss": 0.6931321159029799,
  "accuracy": 0.5223172572414289,
  "precision": 0.5549826252662258,
  "recall": 0.649141208863249,
  "auc": 0.5190096756332404,
  "baseline_accuracy": 0.5481923222541809
}


In [19]:
# Save artifacts used by app.py. The app can load this as a fast realtime inference helper.
torch.save(model.state_dict(), MODEL_DIR / 'best_model.pth')
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')

metadata = {
    'feature_cols': FEATURE_COLS,
    'seq_len': SEQ_LEN,
    'framework': 'pytorch',
    'direction_target_days': DIRECTION_TARGET_DAYS,
    'tickers': sorted(stock_frames.keys()),
    'market_tickers': MARKET_TICKERS,
    'model_type': 'multi_stock_cnn_lstm_direction',
    'validation_metrics': valid_metrics,
    'test_metrics': test_metrics,
}

with open(MODEL_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

history_df.to_csv(MODEL_DIR / 'training_history.csv', index=False)

print('Saved:')
print(MODEL_DIR / 'best_model.pth')
print(MODEL_DIR / 'scaler.pkl')
print(MODEL_DIR / 'metadata.json')


Saved:
model_artifacts/best_model.pth
model_artifacts/scaler.pkl
model_artifacts/metadata.json


In [20]:
# Optional quick single-ticker inference check using the saved model artifacts.
def predict_latest_direction(ticker):
    raw = download_ohlcv(ticker)
    feat = add_features(raw, market_context)
    last = feat[FEATURE_COLS].tail(SEQ_LEN)
    scaled = scaler.transform(last.astype(float))
    tensor = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor)).item()
    return {'ticker': ticker, 'bullish_probability': prob, 'signal': 'BULLISH' if prob >= 0.5 else 'BEARISH'}

predict_latest_direction('NVDA')


{'ticker': 'NVDA',
 'bullish_probability': 0.5121352672576904,
 'signal': 'BULLISH'}